In [5]:
import pandas as pd

df = pd.read_csv(r"C:\Users\ASUS\OneDrive\hr_attrition_analysis\WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [9]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [11]:
print(df["EmployeeCount"].unique())
print(df["StandardHours"].unique())
print(df["Over18"].unique())

[1]
[80]
['Y']


In [13]:
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"])
df.shape

(1470, 31)

In [15]:
import numpy as np

# 1. Tenure buckets - groups employees by how long they've been at the company
bins = [-1, 2, 5, 10, 50]
labels = ["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]
df["TenureGroup"] = pd.cut(df["YearsAtCompany"], bins=bins, labels=labels)

# 2. Income band - segments employees into salary tiers
df["IncomeBand"] = pd.qcut(df["MonthlyIncome"], q=4, labels=["Low", "Medium", "High", "Very High"])

# 3. Overworked flag - employees doing overtime AND reporting poor work-life balance
df["IsOverworkedRisk"] = ((df["OverTime"] == "Yes") & (df["WorkLifeBalance"] <= 2)).astype(int)

df[["TenureGroup", "IncomeBand", "IsOverworkedRisk"]].head()

,TenureGroup,IncomeBand,IsOverworkedRisk
0,6-10 yrs,High,1
1,6-10 yrs,High,0
2,0-2 yrs,Low,0
3,6-10 yrs,Low,0
4,0-2 yrs,Medium,0


In [20]:
!pip install pyodbc sqlalchemy --quiet

In [22]:
import pyodbc
import sqlalchemy
print("pyodbc version:", pyodbc.version)
print("sqlalchemy version:", sqlalchemy.__version__)

pyodbc version: 5.0.1
sqlalchemy version: 2.0.30


In [24]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server']


In [28]:
from sqlalchemy import create_engine, text
import urllib

server = 'localhost\\SQLEXPRESS'
database = 'HR_Attrition_DB'

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE=master;"
    f"Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", isolation_level="AUTOCOMMIT")

with engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE {database}"))
    print(f"Database '{database}' created successfully.")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_2540\320624630.py:16: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


Database 'HR_Attrition_DB' created successfully.


In [30]:
# Connect directly to the new database this time (not master)
params2 = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

engine2 = create_engine(f"mssql+pyodbc:///?odbc_connect={params2}")

df.to_sql("employees", engine2, if_exists="replace", index=False)
print("Data pushed to SQL Server successfully.")

C:\Users\ASUS\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Data pushed to SQL Server successfully.


In [32]:
query = """
SELECT COUNT(*) AS total_employees,
       SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS total_attrition,
       ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM employees
"""

result = pd.read_sql(query, engine2)
result

,total_employees,total_attrition,attrition_rate_pct
0,1470,237,16.1


In [34]:
query2 = """
SELECT Department,
       COUNT(*) AS total_employees,
       SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
       ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM employees
GROUP BY Department
ORDER BY attrition_rate_pct DESC
"""

result2 = pd.read_sql(query2, engine2)
result2

,Department,total_employees,attrition_count,attrition_rate_pct
0,Sales,446,92,20.6
1,Human Resources,63,12,19.0
2,Research & Development,961,133,13.8


In [36]:
query3 = """
SELECT OverTime,
       COUNT(*) AS total_employees,
       SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
       ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM employees
GROUP BY OverTime
ORDER BY attrition_rate_pct DESC
"""

result3 = pd.read_sql(query3, engine2)
result3

,OverTime,total_employees,attrition_count,attrition_rate_pct
0,Yes,416,127,30.5
1,No,1054,110,10.4


In [38]:
query4 = """
SELECT IsOverworkedRisk,
       COUNT(*) AS total_employees,
       SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
       ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM employees
GROUP BY IsOverworkedRisk
ORDER BY attrition_rate_pct DESC
"""

result4 = pd.read_sql(query4, engine2)
result4

,IsOverworkedRisk,total_employees,attrition_count,attrition_rate_pct
0,1,126,42,33.3
1,0,1344,195,14.5


In [40]:
query5 = """
SELECT IncomeBand,
       COUNT(*) AS total_employees,
       SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
       ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM employees
GROUP BY IncomeBand
ORDER BY attrition_rate_pct DESC
"""

result5 = pd.read_sql(query5, engine2)
result5

,IncomeBand,total_employees,attrition_count,attrition_rate_pct
0,Low,369,108,29.3
1,Medium,366,52,14.2
2,High,367,39,10.6
3,Very High,368,38,10.3


In [44]:
query6 = """
SELECT JobRole,
       Attrition,
       MonthlyIncome,
       RANK() OVER (PARTITION BY JobRole ORDER BY MonthlyIncome DESC) AS income_rank_in_role
FROM employees
WHERE Attrition = 'Yes'
"""

result6 = pd.read_sql(query6, engine2)
result6.head(15)

,JobRole,Attrition,MonthlyIncome,income_rank_in_role
0,Healthcare Representative,Yes,12169,1
1,Healthcare Representative,Yes,10312,2
2,Healthcare Representative,Yes,9824,3
3,Healthcare Representative,Yes,8926,4
4,Healthcare Representative,Yes,8722,5
5,Healthcare Representative,Yes,7978,6
6,Healthcare Representative,Yes,7553,7
7,Healthcare Representative,Yes,6673,8
8,Healthcare Representative,Yes,4777,9
9,Human Resources,Yes,10482,1


In [5]:
import pandas as pd

df = pd.read_csv(r"C:\Users\ASUS\OneDrive\hr_attrition_analysis\WA_Fn-UseC_-HR-Employee-Attrition.csv")
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"])

import numpy as np
bins = [-1, 2, 5, 10, 50]
labels = ["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]
df["TenureGroup"] = pd.cut(df["YearsAtCompany"], bins=bins, labels=labels)

df["IncomeBand"] = pd.qcut(df["MonthlyIncome"], q=4, labels=["Low", "Medium", "High", "Very High"])

df["IsOverworkedRisk"] = ((df["OverTime"] == "Yes") & (df["WorkLifeBalance"] <= 2)).astype(int)

In [11]:
bins_age = [18, 25, 35, 45, 60]
labels_age = ["18-25", "26-35", "36-45", "46-60"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins_age, labels=labels_age)

In [13]:
from sqlalchemy import create_engine, text
import urllib

server = 'localhost\\SQLEXPRESS'
database = 'HR_Attrition_DB'

params2 = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

engine2 = create_engine(f"mssql+pyodbc:///?odbc_connect={params2}")

df.to_sql("employees", engine2, if_exists="replace", index=False)
print("Data updated in SQL Server successfully.")

C:\Users\ASUS\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Data updated in SQL Server successfully.


In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import urllib

# Reload and clean data
df = pd.read_csv(r"C:\Users\ASUS\OneDrive\hr_attrition_analysis\WA_Fn-UseC_-HR-Employee-Attrition.csv")
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"])

# All feature engineering
bins = [-1, 2, 5, 10, 50]
labels = ["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]
df["TenureGroup"] = pd.cut(df["YearsAtCompany"], bins=bins, labels=labels)
df["IncomeBand"] = pd.qcut(df["MonthlyIncome"], q=4, labels=["Low", "Medium", "High", "Very High"])
df["IsOverworkedRisk"] = ((df["OverTime"] == "Yes") & (df["WorkLifeBalance"] <= 2)).astype(int)

# Fixed AgeGroup (no blanks)
bins_age = [17, 25, 35, 45, 100]
labels_age = ["18-25", "26-35", "36-45", "46+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins_age, labels=labels_age)

# Reconnect and push to SQL Server
server = 'localhost\\SQLEXPRESS'
database = 'HR_Attrition_DB'
params2 = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)
engine2 = create_engine(f"mssql+pyodbc:///?odbc_connect={params2}")
df.to_sql("employees", engine2, if_exists="replace", index=False)
print("Done — data updated with fixed AgeGroup.")

C:\Users\ASUS\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Done — data updated with fixed AgeGroup.
